In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# %% [markdown]
# # 03 - News collection: GDELT sentiment + SEC EDGAR filings
# Collects a market-wide daily tone index (GDELT) and firm-specific filing
# events (SEC EDGAR) for 5 tickers, keyed on `date_only` to match
# all_tickers_2020_2025.csv. Runs fully on HTTP requests - no GCP/BigQuery
# credentials needed, so it works as-is on Kaggle with internet access
# enabled in notebook settings (Settings -> Internet -> On).

In [3]:
import json
import random
import time
from pathlib import Path

import pandas as pd
import requests

In [4]:
import requests
import pandas as pd
import time

url = "https://api.gdeltproject.org/api/v2/doc/doc"

ticker = "NVDA"
company = '"NVIDIA"'

start_year = 2020
end_year = 2025

all_rows = []

for year in range(start_year, end_year + 1):

    print(f"\nFetching {ticker} news tone for {year}...")

    params = {
        "query": company,
        "mode": "timelinetone",
        "format": "json",
        "startdatetime": f"{year}0101000000",
        "enddatetime": f"{year}1231235959",
    }

    success = False

    for attempt in range(5):

        resp = requests.get(
            url,
            params=params,
            timeout=60,
            headers={"User-Agent": "Mozilla/5.0 research-project/1.0"}
        )

        print("Status:", resp.status_code)

        if resp.status_code == 200:

            data = resp.json()

            for series in data.get("timeline", []):

                for point in series.get("data", []):

                    all_rows.append({
                        "date_only": point["date"][:8],
                        "tone": point["value"],
                        "ticker": ticker
                    })

            print(f"Finished {year}")
            success = True
            break

        elif resp.status_code == 429:

            wait_time = 30 + attempt * 30

            print(
                f"Rate limit reached. "
                f"Waiting {wait_time} seconds before retrying {year}..."
            )

            time.sleep(wait_time)

        else:

            print("Request failed:", resp.text[:200])
            time.sleep(30)

    if not success:
        print(f"Could not collect {year} after 5 attempts.")

    # Extra gap before starting the next year
    time.sleep(15)


nvda_gdelt = pd.DataFrame(all_rows)

if not nvda_gdelt.empty:

    nvda_gdelt["date_only"] = pd.to_datetime(
        nvda_gdelt["date_only"],
        format="%Y%m%d"
    )

    nvda_gdelt = nvda_gdelt.drop_duplicates(
        subset=["ticker", "date_only"]
    )

    nvda_gdelt.to_csv(
        "gdelt_nvda_2020_2025.csv",
        index=False
    )

    print("\nSaved gdelt_nvda_2020_2025.csv")
    print(nvda_gdelt.shape)

    display(nvda_gdelt.head())
    display(nvda_gdelt.tail())

else:

    print("\nNo data were collected.")


Fetching NVDA news tone for 2020...
Status: 429
Rate limit reached. Waiting 30 seconds before retrying 2020...
Status: 429
Rate limit reached. Waiting 60 seconds before retrying 2020...
Status: 429
Rate limit reached. Waiting 90 seconds before retrying 2020...
Status: 429
Rate limit reached. Waiting 120 seconds before retrying 2020...
Status: 200
Finished 2020

Fetching NVDA news tone for 2021...
Status: 200
Finished 2021

Fetching NVDA news tone for 2022...
Status: 200
Finished 2022

Fetching NVDA news tone for 2023...
Status: 429
Rate limit reached. Waiting 30 seconds before retrying 2023...
Status: 429
Rate limit reached. Waiting 60 seconds before retrying 2023...
Status: 429
Rate limit reached. Waiting 90 seconds before retrying 2023...
Status: 429
Rate limit reached. Waiting 120 seconds before retrying 2023...
Status: 429
Rate limit reached. Waiting 150 seconds before retrying 2023...
Could not collect 2023 after 5 attempts.

Fetching NVDA news tone for 2024...
Status: 429
Rate l

,date_only,tone,ticker
0,2020-01-01,1.0359,NVDA
1,2020-01-02,2.2550,NVDA
2,2020-01-03,1.4056,NVDA
3,2020-01-04,1.7423,NVDA
4,2020-01-05,1.8434,NVDA


,date_only,tone,ticker
1438,2025-12-27,1.0763,NVDA
1439,2025-12-28,0.8075,NVDA
1440,2025-12-29,0.5789,NVDA
1441,2025-12-30,0.2364,NVDA
1442,2025-12-31,-0.2209,NVDA


In [5]:
import requests
import pandas as pd
import time
import os

url = "https://api.gdeltproject.org/api/v2/doc/doc"

ticker = "NVDA"
company = '"NVIDIA"'

year = 2022
output_file = "gdelt_nvda_2020_2025.csv"

all_rows = []

print(f"Fetching {ticker} news tone for {year}...")

params = {
    "query": company,
    "mode": "timelinetone",
    "format": "json",
    "startdatetime": f"{year}0101000000",
    "enddatetime": f"{year}1231235959",
}

success = False

for attempt in range(5):

    resp = requests.get(
        url,
        params=params,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0 research-project/1.0"}
    )

    print("Status:", resp.status_code)

    if resp.status_code == 200:

        data = resp.json()

        for series in data.get("timeline", []):
            for point in series.get("data", []):
                all_rows.append({
                    "date_only": point["date"][:8],
                    "tone": point["value"],
                    "ticker": ticker
                })

        print(f"Finished {year}")
        success = True
        break

    elif resp.status_code == 429:

        wait_time = 30 + attempt * 30

        print(
            f"Rate limit reached. "
            f"Waiting {wait_time} seconds before retrying {year}..."
        )

        time.sleep(wait_time)

    else:

        print("Request failed:", resp.text[:200])
        time.sleep(30)


if success:

    gdelt_2022 = pd.DataFrame(all_rows)

    gdelt_2022["date_only"] = pd.to_datetime(
        gdelt_2022["date_only"],
        format="%Y%m%d"
    )

    if os.path.exists(output_file):

        existing_df = pd.read_csv(output_file)

        existing_df["date_only"] = pd.to_datetime(
            existing_df["date_only"]
        )

        combined_df = pd.concat(
            [existing_df, gdelt_2022],
            ignore_index=True
        )

    else:

        combined_df = gdelt_2022.copy()

    combined_df = (
        combined_df
        .drop_duplicates(
            subset=["ticker", "date_only"],
            keep="last"
        )
        .sort_values(["ticker", "date_only"])
        .reset_index(drop=True)
    )

    combined_df.to_csv(
        output_file,
        index=False
    )

    print(f"Added 2022 data to {output_file}")
    print("Total rows:", len(combined_df))

else:

    print("2022 could not be collected.")

Fetching NVDA news tone for 2022...
Status: 429
Rate limit reached. Waiting 30 seconds before retrying 2022...
Status: 429
Rate limit reached. Waiting 60 seconds before retrying 2022...
Status: 429
Rate limit reached. Waiting 90 seconds before retrying 2022...
Status: 429
Rate limit reached. Waiting 120 seconds before retrying 2022...
Status: 429
Rate limit reached. Waiting 150 seconds before retrying 2022...
2022 could not be collected.


In [6]:
check_df = pd.read_csv("gdelt_nvda_2020_2025.csv")

check_df["date_only"] = pd.to_datetime(
    check_df["date_only"]
)

print(
    check_df.groupby(
        check_df["date_only"].dt.year
    ).size()
)

date_only
2020    365
2021    365
2022    365
2025    348
dtype: int64


Inspect and found missing date and data. 

| Year | Expected calendar days | Collected | Missing |
|------|------------------------|-----------|---------|
| 2020 | 366 | 365 | 1 |
| 2021 | 365 | 365 | 0 |
| 2022 | 365 | 365 | 0 |
| 2023 | 365 | 364 | 1 |
| 2024 | 366 | 366 | 0 |
| 2025 | 365 | 348 | 17 |

In [7]:
import pandas as pd

check_df = pd.read_csv("gdelt_nvda_2020_2025.csv")

check_df["date_only"] = pd.to_datetime(
    check_df["date_only"]
)

for year in range(2020, 2026):

    expected_dates = pd.date_range(
        start=f"{year}-01-01",
        end=f"{year}-12-31",
        freq="D"
    )

    collected_dates = check_df.loc[
        check_df["date_only"].dt.year == year,
        "date_only"
    ]

    missing_dates = expected_dates.difference(
        collected_dates
    )

    print(f"\n{year}: {len(missing_dates)} missing dates")

    if len(missing_dates) > 0:
        print(missing_dates.tolist())


2020: 1 missing dates
[Timestamp('2020-10-20 00:00:00')]

2021: 0 missing dates

2022: 0 missing dates

2023: 365 missing dates
[Timestamp('2023-01-01 00:00:00'), Timestamp('2023-01-02 00:00:00'), Timestamp('2023-01-03 00:00:00'), Timestamp('2023-01-04 00:00:00'), Timestamp('2023-01-05 00:00:00'), Timestamp('2023-01-06 00:00:00'), Timestamp('2023-01-07 00:00:00'), Timestamp('2023-01-08 00:00:00'), Timestamp('2023-01-09 00:00:00'), Timestamp('2023-01-10 00:00:00'), Timestamp('2023-01-11 00:00:00'), Timestamp('2023-01-12 00:00:00'), Timestamp('2023-01-13 00:00:00'), Timestamp('2023-01-14 00:00:00'), Timestamp('2023-01-15 00:00:00'), Timestamp('2023-01-16 00:00:00'), Timestamp('2023-01-17 00:00:00'), Timestamp('2023-01-18 00:00:00'), Timestamp('2023-01-19 00:00:00'), Timestamp('2023-01-20 00:00:00'), Timestamp('2023-01-21 00:00:00'), Timestamp('2023-01-22 00:00:00'), Timestamp('2023-01-23 00:00:00'), Timestamp('2023-01-24 00:00:00'), Timestamp('2023-01-25 00:00:00'), Timestamp('2023-01-2

In [8]:
print(
    check_df.groupby(
        check_df["date_only"].dt.year
    )["date_only"].agg(
        ["min", "max", "count"]
    )
)

                 min        max  count
date_only                             
2020      2020-01-01 2020-12-31    365
2021      2021-01-01 2021-12-31    365
2022      2022-01-01 2022-12-31    365
2025      2025-01-01 2025-12-31    348


In [9]:
import requests
import pandas as pd
import time

url = "https://api.gdeltproject.org/api/v2/doc/doc"

params = {
    "query": '"NVIDIA"',
    "mode": "timelinetone",
    "format": "json",
    "startdatetime": "20250615000000",
    "enddatetime": "20250701235959",
}

for attempt in range(5):

    resp = requests.get(
        url,
        params=params,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0 research-project/1.0"}
    )

    print("Status:", resp.status_code)

    if resp.status_code == 200:
        data = resp.json()
        break

    if resp.status_code == 429:
        wait_time = 30 + attempt * 30
        print(f"Rate limited. Waiting {wait_time} seconds...")
        time.sleep(wait_time)

else:
    data = None
    print("Could not collect the missing period.")

Status: 200


In [10]:
missing_rows = []

if data is not None:

    for series in data.get("timeline", []):

        for point in series.get("data", []):

            missing_rows.append({
                "date_only": point["date"][:8],
                "tone": point["value"],
                "ticker": "NVDA"
            })


if len(missing_rows) == 0:

    print("GDELT returned no timeline data for this period.")

else:

    missing_2025 = pd.DataFrame(missing_rows)

    missing_2025["date_only"] = pd.to_datetime(
        missing_2025["date_only"],
        format="%Y%m%d"
    )

    print("Rows collected:", len(missing_2025))

    display(missing_2025.head())

GDELT returned no timeline data for this period.
